# Visualize Alignment Optimization Results

This notebook demonstrates how to load and visualize the optimization data captured during tilt-series alignment.

**Use this for:**
- Creating presentation-quality figures
- Exploring optimization behavior
- Analyzing precision evolution
- Comparing subvolumes across optimization steps

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch

from miss_alignment.alignment.visualize_alignment import load_optimization_data

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load Optimization Data

In [ ]:
# Set path to your optimization tracking directory
tracking_dir = Path("/path/to/output/optimization_steps")

# Load all step data
step_data = load_optimization_data(tracking_dir, load_subvolumes=True)

print(f"Loaded {len(step_data)} optimization steps")
print(f"Initial loss: {step_data[0].loss:.6f}")
print(f"Final loss: {step_data[-1].loss:.6f}")
print(f"Loss reduction: {step_data[0].loss - step_data[-1].loss:.6f}")

## 5. Shift Evolution Analysis

Track how alignment shifts change during optimization.

In [ ]:
if step_data[0].shifts_x is not None:
    # Extract shifts
    shifts_x = torch.stack([s.shifts_x for s in step_data], dim=0)  # (n_steps, n_tilts)
    shifts_y = torch.stack([s.shifts_y for s in step_data], dim=0)
    
    # Compute shift magnitude
    shift_magnitude = torch.sqrt(shifts_x**2 + shifts_y**2)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot mean and individual tilts
    mean_magnitude = shift_magnitude.mean(dim=1)
    ax.plot(steps, mean_magnitude, linewidth=3, color='black', label='Mean', zorder=10)
    
    for i in range(shifts_x.shape[1]):
        ax.plot(steps, shift_magnitude[:, i], alpha=0.2, color='C0', linewidth=1)
    
    ax.set_xlabel('Optimization Step', fontsize=14)
    ax.set_ylabel('Shift Magnitude (Angstroms)', fontsize=14)
    ax.set_title('Shift Evolution During Optimization', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No shift data available")

## 9. Precision-Weighted Subvolume Grid Visualization

Visualize all subvolumes in a grid, weighted by their precision values to show which patches the model is confident about.

In [ ]:
def create_unweighted_grid(subvolumes, projection_dim):
    """Create a grid of unweighted subvolume projections (reference).

    Parameters
    ----------
    subvolumes : torch.Tensor
        Subvolumes tensor of shape (n, d, h, w).
    projection_dim : int
        Dimension to project along (0=depth, 1=height, 2=width).

    Returns
    -------
    grid_image : np.ndarray
        2D grid image suitable for imshow.
    """
    # Mean projection along specified dimension
    projections = subvolumes.mean(dim=projection_dim + 1)  # +1 because first dim is batch

    # Calculate grid dimensions
    n_patches = projections.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_patches)))
    grid_rows = int(np.ceil(n_patches / grid_cols))

    # Get patch size
    patch_h, patch_w = projections.shape[1], projections.shape[2]

    # Create grid
    grid = np.zeros((grid_rows * patch_h, grid_cols * patch_w))

    for idx in range(n_patches):
        row = idx // grid_cols
        col = idx % grid_cols

        patch = projections[idx].numpy()
        grid[row * patch_h:(row + 1) * patch_h,
             col * patch_w:(col + 1) * patch_w] = patch

    return grid


def create_precision_weighted_grid(subvolumes, precisions, projection_dim, normalize_precision=True):
    """Create a grid of precision-weighted subvolume projections.

    Parameters
    ----------
    subvolumes : torch.Tensor
        Subvolumes tensor of shape (n, d, h, w).
    precisions : torch.Tensor
        Precision values of shape (n,).
    projection_dim : int
        Dimension to project along (0=depth, 1=height, 2=width).
    normalize_precision : bool
        Whether to normalize precisions to [0, 1] range.

    Returns
    -------
    grid_image : np.ndarray
        2D grid image suitable for imshow.
    """
    # Mean projection along specified dimension
    projections = subvolumes.mean(dim=projection_dim + 1)  # +1 because first dim is batch

    # Normalize precisions to [0, 1]
    if normalize_precision:
        precision_min = precisions.min()
        precision_max = precisions.max()
        if precision_max > precision_min:
            precisions_norm = (precisions - precision_min) / (precision_max - precision_min)
        else:
            precisions_norm = torch.ones_like(precisions)
    else:
        precisions_norm = precisions

    # Apply precision weighting
    weighted_projections = projections * precisions_norm.view(-1, 1, 1)

    # Calculate grid dimensions
    n_patches = weighted_projections.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_patches)))
    grid_rows = int(np.ceil(n_patches / grid_cols))

    # Get patch size
    patch_h, patch_w = weighted_projections.shape[1], weighted_projections.shape[2]

    # Create grid
    grid = np.zeros((grid_rows * patch_h, grid_cols * patch_w))

    for idx in range(n_patches):
        row = idx // grid_cols
        col = idx % grid_cols

        patch = weighted_projections[idx].numpy()
        grid[row * patch_h:(row + 1) * patch_h,
             col * patch_w:(col + 1) * patch_w] = patch

    return grid


def create_loss_overlay_grid(subvolumes, scores, projection_dim, score_min=None, score_max=None):
    """Create a grid of subvolume projections colored by loss values.

    Parameters
    ----------
    subvolumes : torch.Tensor
        Subvolumes tensor of shape (n, d, h, w).
    scores : torch.Tensor
        Score/loss values of shape (n,).
    projection_dim : int
        Dimension to project along (0=depth, 1=height, 2=width).
    score_min : float, optional
        Minimum score for normalization. If None, computed from scores.
    score_max : float, optional
        Maximum score for normalization. If None, computed from scores.

    Returns
    -------
    grid_image : np.ndarray
        2D grid image suitable for imshow.
    scores_grid : np.ndarray
        Grid of score values for color overlay.
    """
    # Mean projection along specified dimension
    projections = subvolumes.mean(dim=projection_dim + 1)

    # Normalize scores using global or local min/max
    if score_min is None:
        score_min = scores.min()
    if score_max is None:
        score_max = scores.max()

    if score_max > score_min:
        scores_norm = (scores - score_min) / (score_max - score_min)
    else:
        scores_norm = torch.ones_like(scores)

    # Calculate grid dimensions
    n_patches = projections.shape[0]
    grid_cols = int(np.ceil(np.sqrt(n_patches)))
    grid_rows = int(np.ceil(n_patches / grid_cols))

    # Get patch size
    patch_h, patch_w = projections.shape[1], projections.shape[2]

    # Create grids
    grid = np.zeros((grid_rows * patch_h, grid_cols * patch_w))
    scores_grid = np.full((grid_rows * patch_h, grid_cols * patch_w), np.nan)

    for idx in range(n_patches):
        row = idx // grid_cols
        col = idx % grid_cols

        patch = projections[idx].numpy()
        score_val = scores_norm[idx].item()

        grid[row * patch_h:(row + 1) * patch_h,
             col * patch_w:(col + 1) * patch_w] = patch
        scores_grid[row * patch_h:(row + 1) * patch_h,
                   col * patch_w:(col + 1) * patch_w] = score_val

    return grid, scores_grid

### 9.1 Static Precision-Weighted Grid Visualization

Show precision-weighted projections for the first and last optimization steps.

In [ ]:
# Choose initial and final steps
initial_step = step_data[0]
final_step = step_data[-1]

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Create figure for initial and final steps across all projection directions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for proj_dim in range(3):
    # Initial step
    grid_initial = create_precision_weighted_grid(
        initial_step.subvolumes,
        initial_step.precisions,
        projection_dim=proj_dim,
        normalize_precision=True
    )

    axes[0, proj_dim].imshow(grid_initial, cmap='gray', interpolation='bilinear')
    axes[0, proj_dim].set_title(
        f'Initial (Step {initial_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {initial_step.loss:.4f}, Mean Precision: {initial_step.mean_precision:.4f}',
        fontsize=11
    )
    axes[0, proj_dim].axis('off')

    # Final step
    grid_final = create_precision_weighted_grid(
        final_step.subvolumes,
        final_step.precisions,
        projection_dim=proj_dim,
        normalize_precision=True
    )

    axes[1, proj_dim].imshow(grid_final, cmap='gray', interpolation='bilinear')
    axes[1, proj_dim].set_title(
        f'Final (Step {final_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {final_step.loss:.4f}, Mean Precision: {final_step.mean_precision:.4f}',
        fontsize=11
    )
    axes[1, proj_dim].axis('off')

plt.suptitle('Precision-Weighted Subvolume Grid Visualization', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('precision_weighted_grid_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: precision_weighted_grid_comparison.png")

### 9.2 Loss Overlay Grid Visualization

Show subvolume projections with loss values overlaid as a color map.

In [ ]:
# Compute global min/max scores across all steps for consistent normalization
all_scores = torch.cat([s.scores for s in step_data])
global_score_min = all_scores.min().item()
global_score_max = all_scores.max().item()

print(f"Global score range: [{global_score_min:.4f}, {global_score_max:.4f}]")

# Choose initial and final steps
initial_step = step_data[0]
final_step = step_data[-1]

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Create figure for initial and final steps across all projection directions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for proj_dim in range(3):
    # Initial step
    grid_initial, scores_grid_initial = create_loss_overlay_grid(
        initial_step.subvolumes,
        initial_step.scores,
        projection_dim=proj_dim,
        score_min=global_score_min,
        score_max=global_score_max
    )

    # Show grayscale projection with loss overlay
    axes[0, proj_dim].imshow(grid_initial, cmap='gray', interpolation='bilinear', alpha=0.6)
    im0 = axes[0, proj_dim].imshow(scores_grid_initial, cmap='coolwarm', interpolation='bilinear', alpha=0.6)
    axes[0, proj_dim].set_title(
        f'Initial (Step {initial_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {initial_step.loss:.4f}',
        fontsize=11
    )
    axes[0, proj_dim].axis('off')
    cbar0 = plt.colorbar(im0, ax=axes[0, proj_dim], fraction=0.046, pad=0.04)
    cbar0.set_label('Normalized Loss', rotation=270, labelpad=20)

    # Final step
    grid_final, scores_grid_final = create_loss_overlay_grid(
        final_step.subvolumes,
        final_step.scores,
        projection_dim=proj_dim,
        score_min=global_score_min,
        score_max=global_score_max
    )

    axes[1, proj_dim].imshow(grid_final, cmap='gray', interpolation='bilinear', alpha=0.6)
    im1 = axes[1, proj_dim].imshow(scores_grid_final, cmap='coolwarm', interpolation='bilinear', alpha=0.6)
    axes[1, proj_dim].set_title(
        f'Final (Step {final_step.step}) - {projection_names[proj_dim]} Projection\n'
        f'Loss: {final_step.loss:.4f}',
        fontsize=11
    )
    axes[1, proj_dim].axis('off')
    cbar1 = plt.colorbar(im1, ax=axes[1, proj_dim], fraction=0.046, pad=0.04)
    cbar1.set_label('Normalized Loss', rotation=270, labelpad=20)

plt.suptitle('Loss Overlay on Subvolume Grid (Globally Normalized)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('loss_overlay_grid_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: loss_overlay_grid_comparison.png")

### 9.3 Animated Precision-Weighted Grid

Create animated GIFs showing how precision-weighted subvolume projections evolve during optimization.

In [ ]:
import io
from PIL import Image

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Generate two-frame comparison animations for each projection direction
for proj_dim in range(3):
    print(f"Generating precision-weighted comparison animation for {projection_names[proj_dim]} projection...")

    # We'll create two-frame GIFs for initial and final steps
    for step_label, step_data_item in [('initial', step_data[0]), ('final', step_data[-1])]:
        # Create reference (unweighted) grid
        grid_reference = create_unweighted_grid(
            step_data_item.subvolumes,
            projection_dim=proj_dim
        )

        # Create precision-weighted grid
        grid_weighted = create_precision_weighted_grid(
            step_data_item.subvolumes,
            step_data_item.precisions,
            projection_dim=proj_dim,
            normalize_precision=True
        )

        # Create two frames
        frames = []

        # Frame 1: Reference (unweighted)
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(grid_reference, cmap='gray', interpolation='bilinear')
        ax.set_title(
            f'Reference (Unweighted) - Step {step_data_item.step}\n'
            f'{projection_names[proj_dim]} Projection\n'
            f'Loss: {step_data_item.loss:.4f}',
            fontsize=14,
            fontweight='bold'
        )
        ax.axis('off')
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img = Image.open(buf)
        frames.append(img.copy())
        buf.close()
        plt.close(fig)

        # Frame 2: Precision-weighted
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(grid_weighted, cmap='gray', interpolation='bilinear')
        ax.set_title(
            f'Precision-Weighted - Step {step_data_item.step}\n'
            f'{projection_names[proj_dim]} Projection\n'
            f'Loss: {step_data_item.loss:.4f}, Mean Precision: {step_data_item.mean_precision:.4f}',
            fontsize=14,
            fontweight='bold'
        )
        ax.axis('off')
        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img = Image.open(buf)
        frames.append(img.copy())
        buf.close()
        plt.close(fig)

        # Save as two-frame GIF
        if frames:
            output_path = f'precision_weighted_comparison_{projection_names[proj_dim].split()[0].lower()}_{step_label}.gif'
            frames[0].save(
                output_path,
                save_all=True,
                append_images=frames[1:],
                duration=1000,  # 1000ms per frame (1 second)
                loop=0,
                optimize=False,
            )
            print(f"  ✓ Saved: {output_path} (2 frames)")

print("\n✓ All precision-weighted comparison animations complete!")

### 9.4 Animated Loss Overlay Grid

Create animated GIFs showing how loss values across subvolumes evolve during optimization.

In [ ]:
import io
from PIL import Image

# Compute global min/max scores across all steps for consistent normalization
all_scores = torch.cat([s.scores for s in step_data])
global_score_min = all_scores.min().item()
global_score_max = all_scores.max().item()

print(f"Global score range: [{global_score_min:.4f}, {global_score_max:.4f}]\n")

projection_names = ['Depth (Z)', 'Height (Y)', 'Width (X)']

# Generate animations for each projection direction
for proj_dim in range(3):
    print(f"Generating loss overlay animation for {projection_names[proj_dim]} projection...")

    frames = []

    for step_data_item in step_data:
        # Create grid for this step with global normalization
        grid, scores_grid = create_loss_overlay_grid(
            step_data_item.subvolumes,
            step_data_item.scores,
            projection_dim=proj_dim,
            score_min=global_score_min,
            score_max=global_score_max
        )

        # Create figure
        fig, ax = plt.subplots(figsize=(10, 10))

        # Show grayscale background with loss overlay
        ax.imshow(grid, cmap='gray', interpolation='bilinear', alpha=0.6)
        im = ax.imshow(scores_grid, cmap='coolwarm', interpolation='bilinear', alpha=0.6, vmin=0, vmax=1)

        ax.set_title(
            f'{projection_names[proj_dim]} Projection - Step {step_data_item.step}\n'
            f'Loss: {step_data_item.loss:.4f}',
            fontsize=14,
            fontweight='bold'
        )
        ax.axis('off')
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Normalized Loss (Global)', rotation=270, labelpad=20)

        plt.tight_layout()

        # Render to image
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        img = Image.open(buf)
        frames.append(img.copy())
        buf.close()
        plt.close(fig)

    # Save as GIF
    if frames:
        output_path = f'loss_overlay_grid_{projection_names[proj_dim].split()[0].lower()}_evolution.gif'
        frames[0].save(
            output_path,
            save_all=True,
            append_images=frames[1:],
            duration=500,  # 500ms per frame
            loop=0,
            optimize=False,
        )
        print(f"✓ Saved: {output_path} ({len(frames)} frames)")
    else:
        print(f"  No frames generated for {projection_names[proj_dim]} projection")

print("\n✓ All loss overlay animations complete!")

### 9.5 Display Generated Animations

Display the generated GIF animations in the notebook.

In [ ]:
from IPython.display import Image as IPImage, display, HTML
from pathlib import Path

# List of generated GIF files
projection_names = ['depth', 'height', 'width']
gif_types = ['precision_weighted_grid', 'loss_overlay_grid']

print("Generated Animations:\n")

for gif_type in gif_types:
    print(f"\n{'='*60}")
    print(f"{gif_type.replace('_', ' ').title()}")
    print('='*60)

    for proj_name in projection_names:
        gif_path = f'{gif_type}_{proj_name}_evolution.gif'

        if Path(gif_path).exists():
            print(f"\n{proj_name.capitalize()} Projection:")
            display(IPImage(filename=gif_path))
        else:
            print(f"\n{proj_name.capitalize()} Projection: File not found ({gif_path})")